---
title: PLACEHOLDER Imputation
subtitle: Results for parameter set PLACEHOLDER
date: "9999-12-31"
edit_url: null
---

In [ ]:
from harpy.report.components import StatsBox, image_viewer, extract_metric, print_html, standard_itable
import os


In [ ]:
plotdir = 'plots/'
statsfile = "~/contig1.stats"
model = "pseudohaploid"
usebx = True
bxlimit = 50000
k = 15
s = 20
ngen = 5
extra = "None"


In [ ]:
dataL = open(statsfile, 'r').readlines()
bcf = os.path.basename(statsfile).replace(".stats", ".bcf")
plotdir = os.path.abspath(plotdir)

sn = extract_metric(dataL, "SN").drop(columns = [0,1])
sn.columns = ["Metric", "Number"]
sn['Metric'] = [i.replace("number of", '').replace(":",'') for i in sn['Metric']]
_ = sn.to_dict('list')
summ_stats = dict(zip(*(_.values())))


:::{card} Model Parameters

![](#placeholder_modelstats)
![](#placeholder_extraparams)
:::

In [ ]:
#| label: placeholder_modelstats
(
    StatsBox()
    .add(model, "Model", textcol = "#a998cc")
    .add(str(usebx), "usebx")
    .add(k, "K Parameter")
    .add(s, "S Parameter")
    .add(ngen, "ngen Parameter")
    .add(bxlimit, "bxlimit")
).render()


In [ ]:
#| label: placeholder_extraparams

print_html(f"Extra Parameters: {extra}")


## Samples
For convenience, this report consolidates the various outputs from `STITCH` and `bcftools` into a single document.
This reflects the general information stored in the records of the BCF file **for this contig** associated with this model. Since STITCH
requires exclusively bi-allelic SNPs as input, you should not expect to see any other types of variants (MNPs, indels, _etc._).

In [ ]:
boxes = StatsBox()
for k,v in summ_stats.items():
    _k = k.replace("SNP sites", "SNPs")
    boxes.add(v, _k)
boxes.render()


## Per Sample Metrics
This table contains variant statistics per individual, corresponding with the `PSC`
section on `bcftools stats` output.

In [ ]:
psc = extract_metric(dataL, "PSC").drop(columns = [0,1])
psc.columns = ["Sample", "HomRef", "HomAlt", "Het", "Transitions", "Transversions", "Indels",	"MeanDepth", "Singletons",	"HapRef", "HapAlt", "Missing"]
standard_itable(psc, "impute.snp.stats", fixedcols=1)


## Plots Reported by STITCH
These are the plots and diagnostics made internally by STITCH, aggregated
here for convenience. Many of the figures are quite detailed and you are encouraged to right-click the figures to open them in new tabs so that you may investigate them in better resolution.

::::{tab-set}
:::{tab-item} Chromosome-wide QC
**Top**: Gives -log10 (HWE p-value)

**Middle**: Shows `INFO` score, which is a measure of the confidence the imputation has in its performance

**Bottom**: Shows allele frequency, which can be useful to get a 
sense of the LD structure with the distribution of allele frequencies,
as well as its relationship to imputation performance. Can also be used
to test how various parameter choices affect imputation. For example,
for species that have recently been through bottlenecks, allele frequencies
are often highly correlated locally, visible as multiple nearby SNPs that 
have the same allele frequency. Choices of imputation parameters that increase
the tightness in the spread of these allele frequencies often correlate with
increased imputation performance.[^1]

```{figure} #placeholder_qc_chrom
Embedded from `metricsForPostImputationQCChromosome.jpg`
```

:::
:::{tab-item} Imputation Sample QC
Similar to the previous plots, but plotting each of the three against each
other. Useful for getting an overall sense of the distribution of the
metrics, particularly `INFO`, which can be useful towards thinking about a
threshold for filtering out variants after imputation. For example, in the
middle plot with estimated allele frequency vs info, if you expect your data
to cluster into a small range of allele frequencies, and your data seems
concentrated in those frequencies for high `INFO`, you can think about an
`INFO` score cutoff that allows you to capture most of these variants.[^2]

```{figure} #placeholder_qc_sample
Embedded from `metricsForPostImputationQC.sample.jpg`
```

:::
:::{tab-item} Real vs Estimated
Scatter plot of real allele frequency as estimated from the pileup of
sequencing reads (x-axis) and estimated allele frequency from the
posterior of the model (y-axis). This is per-SNP, the sum of the average
usage of each ancestral haplotype times the probability that ancestral
haplotype emits an alternate base, divided by the number of samples. One
should generally see good agreement between these. Note that these are at
"good SNPs" with `INFO` score > `0.4` and HWE p-value > `1x10^-6` (the 
later criterion in particular might not be appropriate for all settings). [^2]

```{figure} #placeholder_goodonly
Embedded from `r2.goodonly.jpg`
```

:::
:::{tab-item} Expected Haplotypes

Across the region being imputed (x-axis, physical position), either the
fraction (normal version) or log10 of the expected number of haplotypes
each sample carries from each ancestral haplotype. The most important
thing to see in this plot is that there is not drastic, but rather gradual,
movement between the relative proportions of the ancestral haplotypes.
Large swings in ancestral haplotype sums indicate the heuristics are working
sub-optimally (though this could also be interesting biology!). It is also
useful to check here how many ancestral haplotypes are being used, to see if
some could be trimmed, for example if some are very infrequently used, though
this is rare, as STITCH will try to fill them up. [^2]

```{figure} #placeholder_hapsum
Embedded from `hapSum.png`
```

```{figure} #placeholder_hapsum_log
Embedded from `hapSum_log.png`
```

:::
:::{tab-item} Alpha Matrix
Likely not informative for general use. `alphaMat` is the name of the
internal `STITCH` variable that stores the probability of jumping into
an ancestral haplotype conditional on a jump. Un-normalized is with
respect to movement of all samples, while normalized has sum 1. [^2]

```{figure} #placeholder_alphamat
Embedded from `alphaMat.all.png`
```

```{figure} #placeholder_alphamat_normalized
Embedded from `alphaMat.normalized.png`
```
:::
::::

[^1]:
    adapted from the STITCH [documentation](https://github.com/rwdavies/STITCH/blob/5a5ba98442a105548587ed8cafbf00aece212c94/plots.md)

[^2]:
    excerpt from the STITCH [documentation](https://github.com/rwdavies/STITCH/blob/5a5ba98442a105548587ed8cafbf00aece212c94/plots.md)


In [ ]:
#| label: placeholder_qc_chrom
image_viewer("Region", plotdir, "*QCChrom*jpg", sortkey = lambda p: int(p.parents[1].name.split('-')[0]), recursive = True, thing_to_select = "region", scale = 0.5)


In [ ]:
#| label: placeholder_qc_sample

image_viewer("Region", plotdir, "*QC.sample.jpg", sortkey = lambda p: int(p.parents[1].name.split('-')[0]), recursive = True, thing_to_select = "region", scale = 0.5)


In [ ]:
#| label: placeholder_goodonly

image_viewer("Region", plotdir, "*r2.goodonly.jpg", sortkey = lambda p: int(p.parents[1].name.split('-')[0]), recursive = True, thing_to_select = "region", scale = 0.5)


In [ ]:
#| label: placeholder_hapsum_log

image_viewer("Region", plotdir, "*hapSum_log.png", sortkey = lambda p: int(p.parents[1].name.split('-')[0]), recursive = True, thing_to_select = "region", scale = 0.5)


In [ ]:
#| label: placeholder_hapsum

image_viewer("Region", plotdir, "*hapSum.png", sortkey = lambda p: int(p.parents[1].name.split('-')[0]), recursive = True, thing_to_select = "region", scale = 0.5)


In [ ]:
#| label: placeholder_alphamat

image_viewer("Region", plotdir, "*alphaMat.all.png", sortkey = lambda p: int(p.parents[1].name.split('-')[0]), recursive = True, thing_to_select = "region", scale = 0.5)


In [ ]:
#| label: placeholder_alphamat_normalized

image_viewer("Region", plotdir, "*alphaMat.normalized.png", sortkey = lambda p: int(p.parents[1].name.split('-')[0]), recursive = True, thing_to_select = "region", scale = 0.5)